# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rasheed-hammad/machine-learning-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
# Purpose: load the Hugging Face token used to access the FlyRank warehouse.

from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [3]:
# Purpose: download the same March, April, and content metadata
# files used to construct the validated W06 population.

from huggingface_hub import hf_hub_download

march_local = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_local = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

content_local = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March file downloaded.")
print("April file downloaded.")
print("Content metadata downloaded.")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

March file downloaded.
April file downloaded.
Content metadata downloaded.


In [4]:
# Purpose: create a DuckDB connection and aggregate March performance
# into one row per client-content pair, matching the W06 data setup.

import duckdb
import pandas as pd

con = duckdb.connect()

march_page = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        MEDIAN(gsc_avg_position) AS march_avg_position,
        SUM(ga4_pageviews) AS march_pageviews,
        SUM(ga4_sessions) AS march_sessions,
        SUM(ga4_engaged_sessions) AS march_engaged_sessions
    FROM read_parquet('{march_local}')
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

# Calculate March CTR from March clicks and impressions.
march_page["march_ctr"] = (
    march_page["march_clicks"] / march_page["march_impressions"]
).where(
    march_page["march_impressions"] > 0
)

print("March aggregated pages:", len(march_page))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March aggregated pages: 331437


In [5]:
# Purpose: aggregate April GSC performance into one row per
# client-content pair so April can be used only for the outcome.

april_page = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks,
        MEDIAN(gsc_avg_position) AS april_avg_position
    FROM read_parquet('{april_local}')
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

print("April aggregated pages:", len(april_page))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April aggregated pages: 362172


In [6]:
# Purpose: combine March historical features with April outcome data
# using the client-content identifiers.

model_df = march_page.merge(
    april_page,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Merged pages:", len(model_df))

Merged pages: 331436


In [7]:
# Purpose: attach content metadata to the March-April data and keep
# only pages that existed by the March 31 decision point, were published,
# and were not deleted.

content = con.execute(f"""
    SELECT *
    FROM read_parquet('{content_local}')
""").fetchdf()

model_df = model_df.merge(
    content,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Keep only content that was valid at the March 31 decision point.
model_df = model_df[
    (model_df["content_updated_date"].notna()) &
    (pd.to_datetime(model_df["content_updated_date"]) <= pd.Timestamp("2026-03-31")) &
    (model_df["is_published"] == True) &
    (model_df["is_deleted"] == False)
].copy()

# Calculate content age at the March 31 decision point.
model_df["days_since_update"] = (
    pd.Timestamp("2026-03-31")
    - pd.to_datetime(model_df["content_updated_date"])
).dt.days

print("Eligible pages:", len(model_df))
print(
    "Days since update range:",
    model_df["days_since_update"].min(),
    "to",
    model_df["days_since_update"].max()
)

Eligible pages: 37229
Days since update range: 7 to 303


In [8]:
# Purpose: calculate how many April days have GSC data available
# for each client-content pair so we can recreate the audited population.

april_availability = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS april_days,

        -- Number of April days with usable GSC data
        SUM(
            CASE
                WHEN gsc_data_available = TRUE THEN 1
                ELSE 0
            END
        ) AS gsc_available_days

    FROM read_parquet('{april_local}')
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

# Attach the availability information to the modeling data.
model_df = model_df.merge(
    april_availability,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Availability rows:", len(april_availability))
print("Pages after merge:", len(model_df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability rows: 362172
Pages after merge: 37229


In [9]:
# Purpose: recreate the exact audited modeling population used in W06.
# A page must have at least 100 March impressions and at least 20 April
# GSC-available days.

# Calculate the March-to-April impression change.
model_df["impression_change_pct"] = (
    (model_df["april_impressions"] - model_df["march_impressions"])
    / model_df["march_impressions"]
) * 100

# Define the audited eligibility conditions.
model_df["eligible_for_model"] = (
    (model_df["march_impressions"] >= 100) &
    (model_df["gsc_available_days"] >= 20)
)

# Define the decline target only for eligible pages.
model_df["target_declined"] = (
    model_df["eligible_for_model"] &
    (model_df["impression_change_pct"] <= -30)
).astype(int)

# Keep only the audited modeling population.
model_ready_audited = model_df[
    model_df["eligible_for_model"]
].copy()

print("Audited modeling population:", len(model_ready_audited))
print("\nTarget counts:")
print(
    model_ready_audited["target_declined"]
    .value_counts()
    .sort_index()
)

print(
    "\nTarget rate:",
    model_ready_audited["target_declined"].mean()
)

Audited modeling population: 16957

Target counts:
target_declined
0    8919
1    8038
Name: count, dtype: int64

Target rate: 0.47402252756973523


In [10]:
# Purpose: recreate the same five client-grouped folds used in W06.
# Sorting the client IDs makes the assignment reproducible.

from sklearn.model_selection import GroupKFold

# Sort clients so fold assignment is deterministic.
clients = sorted(
    model_ready_audited["client_hash_id"].unique()
)

# Create five validation folds while keeping each client together.
group_kfold = GroupKFold(n_splits=5)

# Start with no fold assigned.
model_ready_audited["cv_fold"] = -1

# Assign each page to the fold of its client.
for fold_number, (_, validation_index) in enumerate(
    group_kfold.split(
        model_ready_audited,
        model_ready_audited["target_declined"],
        groups=model_ready_audited["client_hash_id"]
    ),
    start=1
):
    model_ready_audited.loc[
        model_ready_audited.index[validation_index],
        "cv_fold"
    ] = fold_number

print("Number of clients:", len(clients))
print("Unassigned pages:",
      (model_ready_audited["cv_fold"] == -1).sum())

print("\nPages and clients by fold:")
display(
    model_ready_audited
    .groupby("cv_fold")
    .agg(
        pages=("content_hash_id", "size"),
        clients=("client_hash_id", "nunique"),
        positives=("target_declined", "sum"),
        positive_rate=("target_declined", "mean")
    )
)

Number of clients: 26
Unassigned pages: 0

Pages and clients by fold:


,pages,clients,positives,positive_rate
cv_fold,,,,
1,7473,1,3563,0.476783
2,5804,1,2739,0.471916
3,1227,8,340,0.277099
4,1227,7,751,0.612062
5,1226,9,645,0.526101


In [11]:
# Purpose: define the exact leakage-safe feature set validated in W06.

feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_engaged_sessions",
    "days_since_update",
    "content_type"
]

target_column = "target_declined"

print("Number of model features:", len(feature_columns))
print("Model features:")
print(feature_columns)

Number of model features: 9
Model features:
['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_pageviews', 'march_sessions', 'march_engaged_sessions', 'days_since_update', 'content_type']


In [12]:
# Purpose: recreate the same preprocessing used by the validated W06 model.
# Numeric missing values are imputed with the training median.
# Content type is imputed with its most frequent training value
# and then one-hot encoded.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Numeric features used by the Random Forest.
numeric_features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_engaged_sessions",
    "days_since_update"
]

# Categorical feature used by the Random Forest.
categorical_features = [
    "content_type"
]

# Numeric preprocessing.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing.
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine both preprocessing paths.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [13]:
# Purpose: recreate the exact Random Forest configuration validated in W06.

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Combine preprocessing and Random Forest into one pipeline.
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]
)

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [14]:
# Purpose: generate out-of-fold decline scores for every audited page.
# These scores come only from client-held-out validation folds and will
# be used to rank the W07 action queue.

from sklearn.base import clone

oof_results = []

for fold_number in sorted(model_ready_audited["cv_fold"].unique()):

    # Use all other clients for training.
    train_fold = model_ready_audited[
        model_ready_audited["cv_fold"] != fold_number
    ].copy()

    # Hold out the current clients for validation.
    validation_fold = model_ready_audited[
        model_ready_audited["cv_fold"] == fold_number
    ].copy()

    # Start with a fresh copy of the validated model for this fold.
    fold_model = clone(rf_pipeline)

    # Train only on the training clients.
    fold_model.fit(
        train_fold[feature_columns],
        train_fold[target_column]
    )

    # Predict decline probability for the unseen validation clients.
    decline_scores = fold_model.predict_proba(
        validation_fold[feature_columns]
    )[:, 1]

    # Store the page identifiers, target, and out-of-fold score.
    fold_output = validation_fold[
        [
            "client_hash_id",
            "content_hash_id",
            target_column
        ]
    ].copy()

    fold_output["oof_decline_score"] = decline_scores
    fold_output["cv_fold"] = fold_number

    oof_results.append(fold_output)

# Combine all five validation folds into one out-of-fold table.
oof_queue = pd.concat(
    oof_results,
    ignore_index=True
)

print("Out-of-fold rows:", len(oof_queue))
print(
    "Unique content pages:",
    oof_queue["content_hash_id"].nunique()
)
print(
    "Missing decline scores:",
    oof_queue["oof_decline_score"].isna().sum()
)

Out-of-fold rows: 16957
Unique content pages: 16957
Missing decline scores: 0


In [15]:
# Purpose: build the W07 ranked queue from out-of-fold model scores.
# Higher scores are reviewed first because the model ranked those pages
# more strongly as associated with subsequent impression decline.

# Select the page-level signals that a human reviewer can understand.
queue_columns = [
    "client_hash_id",
    "content_hash_id",
    "march_impressions",
    "march_ctr",
    "march_avg_position",
    "days_since_update",
    "content_type"
]

# Attach those signals to the out-of-fold predictions.
action_queue = oof_queue.merge(
    model_ready_audited[queue_columns],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# Rank pages from highest to lowest out-of-fold decline score.
action_queue = action_queue.sort_values(
    "oof_decline_score",
    ascending=False
).reset_index(drop=True)

# Create a simple review priority based on the ranked score.
action_queue["priority_rank"] = (
    action_queue.index + 1
)

action_queue["review_priority"] = "REVIEW"

print("Action queue rows:", len(action_queue))

print("\nTop 20 ranked pages:")
display(
    action_queue[
        [
            "priority_rank",
            "client_hash_id",
            "content_hash_id",
            "oof_decline_score",
            "march_impressions",
            "march_ctr",
            "march_avg_position",
            "days_since_update",
            "content_type"
        ]
    ].head(20)
)

Action queue rows: 16957

Top 20 ranked pages:


,priority_rank,client_hash_id,content_hash_id,oof_decline_score,march_impressions,march_ctr,march_avg_position,days_since_update,content_type
0,1,client_73cda7b4e4f265ea,content_c1bb25d78e494817,0.990000,1652.0,0.000000,0.083333,34,keyword article
1,2,client_23a62021009f63c4,content_25f3cb645f59f734,0.980000,1250.0,0.000000,0.473684,34,keyword article
2,3,client_73cda7b4e4f265ea,content_3d90018c627e44cb,0.976667,990.0,0.000000,7.162162,34,keyword article
3,4,client_73cda7b4e4f265ea,content_541c9e8400addee2,0.976667,238.0,0.000000,0.000000,34,keyword article
4,5,client_73cda7b4e4f265ea,content_a60c16ffa14ace8e,0.973333,1751.0,0.000000,4.735632,34,keyword article
5,6,client_23a62021009f63c4,content_74a1d007f6a9ef66,0.973333,548.0,0.000000,2.100000,34,keyword article
6,7,client_73cda7b4e4f265ea,content_fb4179ebfe948641,0.973333,986.0,0.000000,5.930233,34,keyword article
7,8,client_e547b89c05043229,content_fdb8d89c198457ee,0.973333,1583.0,0.000000,4.947368,34,keyword article
8,9,client_e547b89c05043229,content_9da95f94b407c3dd,0.973333,315.0,0.000000,3.300000,34,keyword article
9,10,client_23a62021009f63c4,content_ab7b026f01dc4fe6,0.970000,980.0,0.000000,0.500000,34,keyword article


In [16]:
# Purpose: add transparent reason codes that explain why a page
# appears high in the action queue. These codes are decision-support
# signals for human review, not automatic actions.

# High model score: use the top 20% of the out-of-fold queue.
high_risk_threshold = action_queue["oof_decline_score"].quantile(0.80)

# Add the model-risk reason.
action_queue["reason_model"] = action_queue[
    "oof_decline_score"
] >= high_risk_threshold

# Add a visibility reason using the 500-impression threshold used
# in the W04 refresh-priority baseline.
action_queue["reason_visibility"] = (
    action_queue["march_impressions"] >= 500
)

# Add a staleness reason using the 180-day threshold from W04.
action_queue["reason_stale"] = (
    action_queue["days_since_update"] >= 180
)

# Add a ranking reason only when the position is valid.
# avg_position = 0 represents no-data, so it is excluded.
action_queue["reason_ranking"] = (
    (action_queue["march_avg_position"] > 0) &
    (action_queue["march_avg_position"] <= 10)
)

# Convert the four checks into a readable list of reason codes.
def build_reason_codes(row):
    reasons = []

    if row["reason_model"]:
        reasons.append("MODEL_HIGH_RISK")

    if row["reason_visibility"]:
        reasons.append("HIGH_VISIBILITY")

    if row["reason_stale"]:
        reasons.append("STALE_CONTENT")

    if row["reason_ranking"]:
        reasons.append("RANKING_SIGNAL")

    return "; ".join(reasons)

action_queue["reason_codes"] = action_queue.apply(
    build_reason_codes,
    axis=1
)

# Remove the temporary Boolean columns.
action_queue = action_queue.drop(
    columns=[
        "reason_model",
        "reason_visibility",
        "reason_stale",
        "reason_ranking"
    ]
)

print("High-risk score threshold:", high_risk_threshold)

print("\nTop 20 queue with reason codes:")
display(
    action_queue[
        [
            "priority_rank",
            "oof_decline_score",
            "march_impressions",
            "march_avg_position",
            "days_since_update",
            "content_type",
            "reason_codes"
        ]
    ].head(20)
)

High-risk score threshold: 0.6233333333333333

Top 20 queue with reason codes:


,priority_rank,oof_decline_score,march_impressions,march_avg_position,days_since_update,content_type,reason_codes
0,1,0.990000,1652.0,0.083333,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL
1,2,0.980000,1250.0,0.473684,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL
2,3,0.976667,990.0,7.162162,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL
3,4,0.976667,238.0,0.000000,34,keyword article,MODEL_HIGH_RISK
4,5,0.973333,1751.0,4.735632,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL
5,6,0.973333,548.0,2.100000,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL
6,7,0.973333,986.0,5.930233,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL
7,8,0.973333,1583.0,4.947368,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL
8,9,0.973333,315.0,3.300000,34,keyword article,MODEL_HIGH_RISK; RANKING_SIGNAL
9,10,0.970000,980.0,0.500000,34,keyword article,MODEL_HIGH_RISK; HIGH_VISIBILITY; RANKING_SIGNAL


### Archetype → action mapping

The model score determines review priority, but it does not determine the final content action. The reviewer should use the page's observable signals and content type to choose the next step.

| Archetype                       | Typical signals                                                          | Recommended human action                                                                                                                                                           |
| ------------------------------- | ------------------------------------------------------------------------ | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **High-risk + high visibility** | High model score and March impressions ≥500                              | Review first. Check search intent, ranking, title/meta, content quality, and whether the page still satisfies the query before considering a refresh.                              |
| **High-risk + strong ranking**  | High model score and valid average position ≤10                          | Check whether the page is losing clicks despite strong visibility/ranking. Review CTR, search intent alignment, SERP presentation, and competing content before changing the page. |
| **High-risk + stale content**   | High model score and content age ≥180 days                               | Review factual freshness, outdated sections, references, and search intent. Consider a refresh only when the content audit supports it.                                            |
| **High-risk only**              | High model score without strong visibility, ranking, or freshness signal | Investigate the page manually before taking action. The model score alone is not sufficient evidence for a refresh.                                                                |
| **Lower-risk pages**            | Lower model score                                                        | Monitor rather than prioritizing immediate manual review, unless another business or editorial signal justifies attention.                                                         |

No archetype automatically maps to publishing, deletion, rewriting, or other irreversible action. The final decision remains with a human reviewer.


In [18]:
# Purpose: add transparent reason codes to the queue.
# These are review signals, not claims that the model has identified
# a guaranteed future failure.

# Use the 80th percentile only to identify relatively high-scored
# pages within this queue. This is a prioritization threshold, not
# a calibrated probability cutoff.
high_score_threshold = action_queue["oof_decline_score"].quantile(0.80)

def build_reason_codes(row):
    reasons = []

    # Relative model score signal.
    if row["oof_decline_score"] >= high_score_threshold:
        reasons.append("MODEL_HIGH_SCORE")

    # W04 visibility signal.
    if row["march_impressions"] >= 500:
        reasons.append("HIGH_VISIBILITY")

    # W04 freshness/staleness signal.
    if row["days_since_update"] >= 180:
        reasons.append("STALE_CONTENT")

    # Valid page-one ranking signal.
    if (
        row["march_avg_position"] > 0
        and row["march_avg_position"] <= 10
    ):
        reasons.append("RANKING_SIGNAL")

    return "; ".join(reasons)

action_queue["reason_codes"] = action_queue.apply(
    build_reason_codes,
    axis=1
)

print("High-score prioritization threshold:", high_score_threshold)

print("\nTop 20 reason codes:")
display(
    action_queue[
        [
            "priority_rank",
            "oof_decline_score",
            "reason_codes"
        ]
    ].head(20)
)

High-score prioritization threshold: 0.6233333333333333

Top 20 reason codes:


,priority_rank,oof_decline_score,reason_codes
0,1,0.990000,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL
1,2,0.980000,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL
2,3,0.976667,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL
3,4,0.976667,MODEL_HIGH_SCORE
4,5,0.973333,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL
5,6,0.973333,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL
6,7,0.973333,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL
7,8,0.973333,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL
8,9,0.973333,MODEL_HIGH_SCORE; RANKING_SIGNAL
9,10,0.970000,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL


In [19]:
# Purpose: convert each archetype into a human-review recommendation.
# These are suggested next steps only; they do not trigger automatic
# publishing, deletion, or content changes.

def assign_recommended_action(archetype):

    if archetype == "HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING":
        return "PRIORITY_REVIEW"

    if archetype == "HIGH_RISK_HIGH_VISIBILITY":
        return "PRIORITY_REVIEW"

    if archetype == "HIGH_RISK_STRONG_RANKING":
        return "SERP_AND_CTR_REVIEW"

    if archetype == "HIGH_RISK_STALE_CONTENT":
        return "FRESHNESS_REVIEW"

    if archetype == "HIGH_RISK_ONLY":
        return "MANUAL_REVIEW"

    return "MONITOR"

action_queue["recommended_action"] = action_queue[
    "archetype"
].apply(assign_recommended_action)

print("Recommended action counts:")
display(
    action_queue["recommended_action"]
    .value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="pages")
)

print("\nTop 20 action queue:")
display(
    action_queue[
        [
            "priority_rank",
            "oof_decline_score",
            "archetype",
            "reason_codes",
            "recommended_action"
        ]
    ].head(20)
)

Recommended action counts:


,recommended_action,pages
0,MONITOR,13541
1,PRIORITY_REVIEW,2538
2,SERP_AND_CTR_REVIEW,589
3,MANUAL_REVIEW,288
4,FRESHNESS_REVIEW,1



Top 20 action queue:


,priority_rank,oof_decline_score,archetype,reason_codes,recommended_action
0,1,0.990000,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW
1,2,0.980000,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW
2,3,0.976667,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW
3,4,0.976667,HIGH_RISK_ONLY,MODEL_HIGH_SCORE,MANUAL_REVIEW
4,5,0.973333,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW
5,6,0.973333,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW
6,7,0.973333,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW
7,8,0.973333,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW
8,9,0.973333,HIGH_RISK_STRONG_RANKING,MODEL_HIGH_SCORE; RANKING_SIGNAL,SERP_AND_CTR_REVIEW
9,10,0.970000,HIGH_RISK_HIGH_VISIBILITY_STRONG_RANKING,MODEL_HIGH_SCORE; HIGH_VISIBILITY; RANKING_SIGNAL,PRIORITY_REVIEW


In [20]:
# Purpose: summarize observed decline rates by content-age group.
# This is descriptive evidence for the playbook, not a causal estimate
# of what refreshing a page will do.

# Create simple content-age groups at the March 31 decision point.
action_queue["age_group"] = pd.cut(
    action_queue["days_since_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

# Summarize the observed April decline rate in each age group.
age_summary = (
    action_queue
    .groupby("age_group", observed=False)
    .agg(
        pages=("content_hash_id", "size"),
        declining_pages=("target_declined", "sum"),
        decline_rate=("target_declined", "mean")
    )
    .reset_index()
)

display(age_summary)

,age_group,pages,declining_pages,decline_rate
0,0-30 days,66,32,0.484848
1,31-90 days,16762,7951,0.474347
2,91-180 days,118,47,0.398305
3,181+ days,11,8,0.727273


### Decay / refresh insight

Content age is used in the playbook as a review signal, not as proof that older pages will decline or that refreshing a page will improve performance.

In the audited population, most pages were 31–90 days old at the March 31 decision point (16,762 of 16,957 pages), with an observed April decline rate of 47.4%. The 181+ day group had an observed decline rate of 72.7%, but it contained only 11 pages, so this small bucket is not strong enough to support a broad age-effect claim.

The practical use of freshness is therefore diagnostic: when a high-priority page is also old, the reviewer should check for outdated facts, references, search intent, and other evidence that the content needs updating. A refresh should be considered only after that human review.

The earlier FlyRank research also reported higher observed health and impressions for recently refreshed older pages, but those comparisons were observational and may be affected by which pages were selected for refresh. Therefore, the playbook treats refresh as a human decision rather than an automatic consequence of model score or content age.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.